In [1]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms
from cryptography.hazmat.backends import default_backend
import os

In [2]:
def xor_bytes(a, b):
    return bytes(x ^ y for x, y in zip(a, b))

In [3]:
def generate_keystream(key, nonce, length):
    cipher = Cipher(
        algorithms.ChaCha20(key, nonce),
        mode=None,
        backend=default_backend()
    )

    encryptor = cipher.encryptor()

    # Encrypt all-zero bytes to obtain keystream
    keystream = encryptor.update(b'\x00' * length)

    return keystream

In [4]:
def encrypt(plaintext, keystream):
    return xor_bytes(plaintext, keystream[:len(plaintext)])

In [5]:
def decrypt(ciphertext, keystream):
    return xor_bytes(ciphertext, keystream[:len(ciphertext)])

In [8]:
def main():
    key = os.urandom(32)
    nonce = os.urandom(16)
    print("Key:", key.hex())
    print("Nonce:", nonce.hex())
    P1 = b"Attack at dawn!!"
    P2 = b"Send more troops"
    max_len = max(len(P1), len(P2))
    keystream = generate_keystream(key, nonce, max_len)
    print("\nKeystream:")
    print(keystream.hex())
    C1 = encrypt(P1, keystream)
    C2 = encrypt(P2, keystream)
    print("\nCiphertext 1:", C1.hex())
    print("Ciphertext 2:", C2.hex())
    print("\nDecryption:")
    print(decrypt(C1, keystream))
    print(decrypt(C2, keystream))
    print("\n--- Two-Time Pad Attack ---")
    P1_xor_P2 = xor_bytes(C1, C2)
    print("C1 XOR C2 =", P1_xor_P2)
    recovered_P2 = xor_bytes(P1_xor_P2, P1)
    print("\nKnown plaintext P1:")
    print(P1)
    print("\nRecovered P2:")
    print(recovered_P2)
    print("\nActual P2:")
    print(P2)
    if recovered_P2 == P2:
        print("\nAttack Successful!")
    else:
        print("\nAttack Failed.")

In [9]:
main()

Key: f161888fe6338916336522b30ac8241b7d626e9c5dd389b84817e2e22ec0ada8
Nonce: c7754d50ddba9502cd33245259e710cb

Keystream:
61c6a8d3870c9f4d5c6e93c0d41fe611

Ciphertext 1: 20b2dcb2e467bf2c284ef7a1a371c730
Ciphertext 2: 32a3c6b7a761f03f394ee7b2bb709662

Decryption:
b'Attack at dawn!!'
b'Send more troops'

--- Two-Time Pad Attack ---
C1 XOR C2 = b'\x12\x11\x1a\x05C\x06O\x13\x11\x00\x10\x13\x18\x01QR'

Known plaintext P1:
b'Attack at dawn!!'

Recovered P2:
b'Send more troops'

Actual P2:
b'Send more troops'

Attack Successful!
